In [5]:
# Install required packages if running in a fresh environment (e.g. Colab)
# !pip install scikit-learn nltk scipy matplotlib numpy

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from scipy.stats import mode
import numpy as np

# Download necessary NLTK data (only once)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# ────────────────────────────────────────────────
# 1. Load the 20 Newsgroups dataset - only training subset & 5 specific categories
# ────────────────────────────────────────────────
categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.med',
    'sci.space',
    'talk.politics.guns'
]

print("Loading 20 Newsgroups data...")
data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f"Loaded {len(data.data)} documents.")

# ────────────────────────────────────────────────
# 2. Text preprocessing: clean, tokenize, remove stopwords
# ────────────────────────────────────────────────
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize and lowercase
    tokens = word_tokenize(text.lower())
    # Keep only meaningful words (no stop words, no single letters)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

print("Preprocessing texts...")
processed_texts = [clean_text(doc) for doc in data.data]
print("Preprocessing completed.")

# ────────────────────────────────────────────────
# 3. Convert texts to TF-IDF matrix
# ────────────────────────────────────────────────
print("Building TF-IDF matrix...")
vectorizer = TfidfVectorizer(
    max_features=10000,          # limit vocabulary size for speed & memory
    min_df=5,                    # ignore very rare terms
    max_df=0.95                  # ignore very common terms
)

X_tfidf = vectorizer.fit_transform(processed_texts)
X_dense = X_tfidf.toarray()     # convert to dense (required for some clustering algos)

print(f"TF-IDF shape: {X_dense.shape}")

# ────────────────────────────────────────────────
# 4. Clustering
# ────────────────────────────────────────────────

# A. Agglomerative (Ward linkage - good for roughly spherical clusters)
print("Running Agglomerative Clustering...")
agg_model = AgglomerativeClustering(
    n_clusters=5,
    metric='euclidean',          # ← corrected from 'affinity' (scikit-learn ≥1.4)
    linkage='ward'
)
labels_agg = agg_model.fit_predict(X_dense)

# B. DBSCAN (density-based - may produce different number of clusters)
print("Running DBSCAN...")
dbscan_model = DBSCAN(
    eps=1.1,                     # tune this value (try 0.9–1.5)
    min_samples=4,
    metric='euclidean',
    n_jobs=-1
)
labels_db = dbscan_model.fit_predict(X_dense)

# ────────────────────────────────────────────────
# 5. Evaluation
# ────────────────────────────────────────────────
true_labels = data.target

# Agglomerative metrics
ari_agg = adjusted_rand_score(true_labels, labels_agg)
ami_agg = adjusted_mutual_info_score(true_labels, labels_agg)
sil_agg = silhouette_score(X_dense, labels_agg) if len(np.unique(labels_agg)) > 1 else -1

# DBSCAN metrics
ari_db = adjusted_rand_score(true_labels, labels_db)
ami_db = adjusted_mutual_info_score(true_labels, labels_db)
sil_db = silhouette_score(X_dense, labels_db) if len(np.unique(labels_db)) > 1 else -1

print("\n" + "="*50)
print("RESULTS - Agglomerative Clustering")
print(f"ARI:       {ari_agg:.4f}")
print(f"AMI:       {ami_agg:.4f}")
print(f"Silhouette:{sil_agg:.4f}")

print("\nRESULTS - DBSCAN")
print(f"ARI:       {ari_db:.4f}")
print(f"AMI:       {ami_db:.4f}")
print(f"Silhouette:{sil_db:.4f}")
print("="*50 + "\n")

# ────────────────────────────────────────────────
# 6. Cluster → Newsgroup mapping (majority vote)
# ────────────────────────────────────────────────
print("Cluster to category mapping (majority vote):")

print("\nAgglomerative:")
for cluster_id in range(5):
    mask = (labels_agg == cluster_id)
    if np.sum(mask) == 0:
        print(f"Cluster {cluster_id}: empty")
        continue
    majority_idx = mode(true_labels[mask]).mode
    print(f"Cluster {cluster_id} → {categories[majority_idx]} "
          f"({np.sum(mask)} documents)")

print("\nDBSCAN:")
unique_db_labels = np.unique(labels_db)
unique_db_labels = unique_db_labels[unique_db_labels != -1]  # exclude noise

for cluster_id in unique_db_labels:
    mask = (labels_db == cluster_id)
    majority_idx = mode(true_labels[mask]).mode
    print(f"Cluster {cluster_id} → {categories[majority_idx]} "
          f"({np.sum(mask)} documents)")

Loading 20 Newsgroups data...
Loaded 2797 documents.
Preprocessing texts...
Preprocessing completed.
Building TF-IDF matrix...
TF-IDF shape: (2797, 6633)
Running Agglomerative Clustering...
Running DBSCAN...

RESULTS - Agglomerative Clustering
ARI:       0.1078
AMI:       0.2415
Silhouette:0.0118

RESULTS - DBSCAN
ARI:       0.0000
AMI:       0.0000
Silhouette:-1.0000

Cluster to category mapping (majority vote):

Agglomerative:
Cluster 0 → sci.space (2102 documents)
Cluster 1 → sci.med (66 documents)
Cluster 2 → comp.graphics (576 documents)
Cluster 3 → sci.med (39 documents)
Cluster 4 → alt.atheism (14 documents)

DBSCAN:
Cluster 0 → sci.med (2797 documents)
